# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Looking at the raw distributions of the key signals used in my churn framing, before drawing any conclusions from them — impressions_90d, days_since_last_update, and avg_position all tend to have heavy right tails, so a few extreme pages can distort naive averages.

In [1]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)

for col in ["impressions_90d", "days_since_last_update", "avg_position"]:
    print(f"\n{col}:")
    print(df[col].describe())


impressions_90d:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

days_since_last_update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

avg_position:
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Test 1 — Staleness: do stale pages (days_since_last_update >= 180) churn more than fresh pages?
Test 2 — Position: do lower-ranked pages (avg_position > 20) churn more than top-ranked pages?
Test 3 — Word count: do thin pages (word_count < 1200) churn more than long pages?
Each gets a verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [2]:
stale_rate = df[df["days_since_last_update"] >= 180]["is_churned"].mean()
fresh_rate = df[df["days_since_last_update"] < 180]["is_churned"].mean()
print(f"Test 1 - Staleness: stale churn rate={stale_rate:.3f} vs fresh churn rate={fresh_rate:.3f}")
print("Verdict:", "CONFIRMED" if stale_rate > fresh_rate else "OPPOSITE")

low_pos_rate = df[df["avg_position"] > 20]["is_churned"].mean()
high_pos_rate = df[df["avg_position"] <= 20]["is_churned"].mean()
print(f"\nTest 2 - Position: low-rank churn rate={low_pos_rate:.3f} vs top-rank churn rate={high_pos_rate:.3f}")
print("Verdict:", "CONFIRMED" if low_pos_rate > high_pos_rate else "OPPOSITE")

thin_rate = df[df["word_count"] < 1200]["is_churned"].mean()
long_rate = df[df["word_count"] >= 1200]["is_churned"].mean()
print(f"\nTest 3 - Word count: thin churn rate={thin_rate:.3f} vs long churn rate={long_rate:.3f}")
print("Verdict:", "CONFIRMED" if thin_rate > long_rate else "MIXED" if abs(thin_rate-long_rate) < 0.02 else "OPPOSITE")

Test 1 - Staleness: stale churn rate=0.471 vs fresh churn rate=0.542
Verdict: OPPOSITE

Test 2 - Position: low-rank churn rate=0.528 vs top-rank churn rate=0.548
Verdict: OPPOSITE

Test 3 - Word count: thin churn rate=0.250 vs long churn rate=0.588
Verdict: OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's starter reason code "stale_visible_page" assumes that pages which are both stale (>=180 days since update) AND visible (impressions_90d >= 500) are worth reviewing. Testing whether this combined signal actually predicts churn better than either signal alone.

In [3]:
combo = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
combo_rate = df[combo]["is_churned"].mean()
overall_rate = df["is_churned"].mean()
print(f"Combined 'stale_visible_page' rule churn rate: {combo_rate:.3f} (n={combo.sum()})")
print(f"Overall base churn rate: {overall_rate:.3f}")
print("Verdict:", "CONFIRMED" if combo_rate > overall_rate else "FALSE")

Combined 'stale_visible_page' rule churn rate: 0.941 (n=17)
Overall base churn rate: 0.542
Verdict: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should trust the staleness+visibility flag as a starting point, but not as the whole story — it flags very few pages (a tiny slice of the total), so relying on it alone would miss most declining pages. It's precise but not comprehensive, which is exactly the gap a learned model can help close.

In [4]:
print(f"Rule coverage: {combo.sum()} of {len(df)} pages ({combo.mean()*100:.2f}%)")
print("Take-away: precise but narrow — good starting signal, not a complete solution.")

Rule coverage: 17 of 30000 pages (0.06%)
Take-away: precise but narrow — good starting signal, not a complete solution.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.